In [ ]:
import os
import cv2
import json
import re
import torch
import numpy as np
import torch.nn as nn
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torchvision import transforms, models

from torchvision import transforms

inference_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
# Force matplotlib to display plots inline cleanly inside Jupyter cells
%matplotlib inline

# Gracefully handle SAM imports to prevent runtime crashes if not installed
try:
    from segment_anything import sam_model_registry, SamPredictor
    SAM_AVAILABLE = True
except ImportError:
    SAM_AVAILABLE = False

# ============================================================
# 1. HARDWARE & DIRECTORY ARCHITECTURE CONFIGURATIONS
# ============================================================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---> UPDATE THESE PATHS TO MATCH YOUR LOCAL ENVIRONMENT DIRECTORIES <---
MAPPING_PATH = r"C:\Users\moham\OneDrive\Desktop\grad-documentation\project\H2A_cv\classifier\weights\class_mapping.json"
CLASSIFIER_WEIGHTS = r"C:\Users\moham\OneDrive\Desktop\grad-documentation\project\H2A_cv\classifier\weights\best_resnet50_hieroglyph.pth" 
SAM_CHECKPOINT = r"C:\Users\moham\OneDrive\Desktop\grad-documentation\project\H2A_cv\detector\sam_weights\sam_vit_h_4b8939.pth"
TARGET_IMAGE = r"C:\Users\moham\OneDrive\Desktop\grad-documentation\project\H2A_cv\sc2.png"
LEXICON_PATH = r"C:\Users\moham\OneDrive\Desktop\grad-documentation\project\H2A_cv\Gardiner2transliteration\Resources\Lexicon.txt"

print(f"Using execution hardware device: {str(DEVICE).upper()}")

# ============================================================
# 2. DYNAMIC MAP LOADER & ALPHABETICAL BACKSTOPS
# ============================================================
def load_class_mapping_from_json(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        loaded_data = json.load(f)
    return {int(k): v for k, v in loaded_data.items()}

idx_to_class = load_class_mapping_from_json(MAPPING_PATH)
num_classes = 310

# ============================================================
# 3. COMPATIBLE LEXICON STRUCTURE RE-ROUTER
# ============================================================
class LexiconCodeMapper:
    def __init__(self):
        """
        Maps raw predicted vision codes to cross-walk vocabulary keys
        and handles character defaults safely without producing 'UNK' loops.
        """
        self.remap_dictionary = {
            "AA2": "D26", "AA5": "O4", "AA11": "J11", "AA12": "J11",
            "AA13": "U2", "AA15": "J13", "AA27": "J27", "AA28": "J28",
            "J3": "V13", "G38": "G39", "V2": "U30",
        }
        self.phonetic_defaults = {
            "G1": "ꜣ", "M17": "j", "A1": "j", "A40": "j", "A50": "j",
            "D36": "a", "G43": "w", "W24": "w", "D58": "b", "Q3": "p",
            "I9": "f", "G17": "m", "N35": "n", "D21": "r", "O4": "h",
            "V28": "H", "J1": "x", "F32": "X", "O34": "s", "S29": "s",
            "N37": "S", "N29": "q", "V31": "k", "W11": "g", "X1": "t",
            "V13": "T", "D46": "d", "I10": "D", "G25": "Ax", "V16": "sA"
        }

    def map_code(self, raw_sign):
        clean_sign = str(raw_sign).strip().upper()
        if clean_sign in self.remap_dictionary:
            return self.remap_dictionary[clean_sign]
        return clean_sign

    def get_phonetic_fallback(self, clean_sign):
        return self.phonetic_defaults.get(clean_sign.upper(), f"[{clean_sign.upper()}]")

# ============================================================
# 4. ADVANCED SEGMENTATION ENGINE (IGSM using SAM)
# ============================================================
class HieroglyphSegmenter:
    def __init__(self, sam_checkpoint_path, config=None, device=None):
        if not SAM_AVAILABLE:
            raise ImportError("The 'segment_anything' library is missing! Install via pip.")
        self.device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.config = config or {
            'min_area': 250, 'max_area': 15000,
            'min_width': 10, 'min_height': 10,
            'max_width': 180, 'max_height': 180,
            'max_aspect_ratio': 6.0, 'min_aspect_ratio': 0.10,
            'min_solidity': 0.10, 'min_extent': 0.10,
            'crop_padding': 8, 'line_threshold': 55,
        }
        sam = sam_model_registry["vit_h"](checkpoint=sam_checkpoint_path).to(self.device).eval()
        self.predictor = SamPredictor(sam)

    def _generate_point_prompts(self, gray_img):
        blurred = cv2.GaussianBlur(gray_img, (5, 5), 0)
        binary_mask = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 21, 8)
        horizontal_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 1))
        horizontal_lines = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, horizontal_kernel, iterations=2)
        binary_mask = cv2.subtract(binary_mask, horizontal_lines)
        contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        point_prompts = []
        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            if cv2.contourArea(cnt) < self.config['min_area'] * 0.4: continue
            if (w / float(max(h, 1))) > 3.5 or (w / float(max(h, 1))) < 0.2: continue
            moments = cv2.moments(cnt)
            if moments["m00"] != 0:
                point_prompts.append([int(moments["m10"] / moments["m00"]), int(moments["m01"] / moments["m00"])])
        return point_prompts

    def _predict_sam_masks(self, image_rgb, point_prompts):
        h, w = image_rgb.shape[:2]
        unified_mask = np.zeros((h, w), dtype=np.uint8)
        if not point_prompts: return unified_mask
        self.predictor.set_image(image_rgb)
        with torch.no_grad():
            for point in point_prompts:
                masks, scores, _ = self.predictor.predict(point_coords=np.array([point], dtype=np.float32), point_labels=np.array([1], dtype=np.int32), multimask_output=True)
                best_mask, best_score = None, -1
                for i in range(len(masks)):
                    y_idx, x_idx = np.where(masks[i] > 0)
                    if len(y_idx) == 0: continue
                    if (np.count_nonzero(masks[i]) < self.config['max_area'] and (x_idx.max() - x_idx.min()) < self.config['max_width'] and (y_idx.max() - y_idx.min()) < self.config['max_height'] and scores[i] > best_score):
                        best_mask, best_score = masks[i], scores[i]
                if best_mask is not None:
                    unified_mask = cv2.bitwise_or(unified_mask, (best_mask.astype(np.uint8) * 255))
        return unified_mask

    def _sort_boxes_by_reading_order(self, boxes):
        if not boxes: return []
        boxes_array = np.array(boxes)
        centers_y = boxes_array[:, 1] + boxes_array[:, 3] / 2
        sorted_indices = np.argsort(centers_y)
        lines, current_line = [], [sorted_indices[0]]
        current_y = centers_y[sorted_indices[0]]

        for idx in sorted_indices[1:]:
            if abs(centers_y[idx] - current_y) < self.config['line_threshold']:
                current_line.append(idx)
            else:
                current_line.sort(key=lambda i: boxes_array[i, 0])
                lines.append(current_line)
                current_line, current_y = [idx], centers_y[idx]
        if current_line:
            current_line.sort(key=lambda i: boxes_array[i, 0])
            lines.append(current_line)

        return [tuple(boxes_array[idx].astype(int)) for line in lines for idx in line]

    def extract_symbols(self, image_path):
        img_bgr = cv2.imread(image_path)
        h_img, w_img = img_bgr.shape[:2]
        gray_img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        point_prompts = self._generate_point_prompts(gray_img)
        unified_mask = self._predict_sam_masks(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), point_prompts)
        final_contours, _ = cv2.findContours(unified_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        valid_boxes = []
        for cnt in final_contours:
            x, y, w, h = cv2.boundingRect(cnt)
            if (self.config['min_area'] <= cv2.contourArea(cnt) <= self.config['max_area']) and (self.config['min_width'] <= w <= self.config['max_width']) and (self.config['min_height'] <= h <= self.config['max_height']):
                valid_boxes.append((x, y, w, h))
        sorted_boxes = self._sort_boxes_by_reading_order(valid_boxes)
        crops = []
        pad = self.config['crop_padding']
        for x, y, w, h in sorted_boxes:
            crops.append((img_bgr[max(0, y - pad):min(h_img, y + h + pad), max(0, x - pad):min(w_img, x + w + pad)], (x, y, w, h)))
        return img_bgr, unified_mask, crops, len(point_prompts), len(valid_boxes)

# ============================================================
# 5. MODEL LOGICHEAD & CLASSIFICATION INTERCEPTOR
# ============================================================
class HieroglyphClassifier:
    def __init__(self, model, idx_to_class, transform):
        self.model = model
        self.idx_to_class = idx_to_class
        self.transform = transform
        self.mapper = LexiconCodeMapper()
        self.device = next(model.parameters()).device

    def classify(self, crop_bgr):
        pil_crop = Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
        tensor = self.transform(pil_crop).unsqueeze(0).to(self.device)
        with torch.no_grad():
            outputs = self.model(tensor)
            probabilities = torch.softmax(outputs, dim=1)
            conf, pred = torch.max(probabilities, 1)
        return {'label': self.mapper.map_code(self.idx_to_class[pred.item()]), 'confidence': conf.item()}

# ============================================================
# 6. EXACT EXPERT CORPUS LOOKUP & TRANSLITERATION ENGINE
# ============================================================
class ExactCorpusMatchPipeline:
    def __init__(self, lexicon_path):
        self.lexicon_path = lexicon_path
        self.dictionary = {}
        self.mapper = LexiconCodeMapper()
        self.build_model_ready_dictionary()

    def clean_and_align_to_corpus(self, raw_translit: str) -> str:
        """Strips syntax layouts and standardizes character-level outputs (.char Directly)"""
        cleaned = raw_translit.strip().lower()
        cleaned = cleaned.replace('i', 'j').replace('z', 's')
        cleaned = cleaned.replace('=', ' ').replace('.', ' ').replace('-', ' ').replace('*', '')
        cleaned = cleaned.replace('\r\n', ' ').replace('\n', ' ')
        return re.sub(r'\s{2,}', ' ', cleaned).strip()

    def build_model_ready_dictionary(self):
        print(f"Compiling raw target database from {self.lexicon_path}...")
        with open(self.lexicon_path, 'r', encoding='utf-8') as f:
            for line in f:
                if not line.strip(): continue
                parts = line.split(';')
                if len(parts) >= 3:
                    tokens = [self.mapper.map_code(t.strip()) for t in parts[0].split(',') if t.strip()]
                    gardiner_key = " ".join(tokens)
                    model_ready_translit = self.clean_and_align_to_corpus(parts[1])
                    if gardiner_key and model_ready_translit:
                        if (gardiner_key not in self.dictionary) or (len(model_ready_translit) < len(self.dictionary[gardiner_key]["model_input"])):
                            self.dictionary[gardiner_key] = {"model_input": model_ready_translit, "translation_hint": parts[2].strip()}

    def process_gardiner_list(self, extracted_list: list) -> dict:
        """Processes input lists of Gardiner codes into dot-free OpenNMT tokens"""
        cleaned_input_tokens = [self.mapper.map_code(item) for item in extracted_list if str(item).strip()]
        full_query_key = " ".join(cleaned_input_tokens)
        
        if full_query_key in self.dictionary:
            return {"strategy": "exact_phrase_match", "open_nmt_ready_string": self.dictionary[full_query_key]["model_input"], "english_hint": self.dictionary[full_query_key]["translation_hint"]}
            
        resolved_tokens, resolved_hints, i = [], [], 0
        while i < len(cleaned_input_tokens):
            matched = False
            for width in range(min(4, len(cleaned_input_tokens) - i), 0, -1):
                chunk = " ".join(cleaned_input_tokens[i:i+width])
                if chunk in self.dictionary:
                    resolved_tokens.append(self.dictionary[chunk]["model_input"])
                    resolved_hints.append(self.dictionary[chunk]["translation_hint"])
                    i += width
                    matched = True
                    break
            if not matched:
                resolved_tokens.append(self.mapper.get_phonetic_fallback(cleaned_input_tokens[i]))
                i += 1
                
        return {
            "strategy": "sliding_window_fallback",
            "open_nmt_ready_string": re.sub(r'\s{2,}', ' ', " ".join(resolved_tokens)).strip(),
            "english_hint": " / ".join(resolved_hints) if resolved_hints else "Unknown Context"
        }

# ============================================================
# 7. MAIN INTEGRATED JUPYTER CELL RUNNER
# ============================================================
if __name__ == "__main__":
    def build_model(num_classes):
        m = models.resnet50(weights=None)
        m.fc = nn.Sequential(nn.Identity(), nn.Linear(m.fc.in_features, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(p=0.3), nn.Linear(512, num_classes))
        return m

    # 1. Initialize Complete Model Frameworks
    resnet_model = build_model(num_classes)
    resnet_model.load_state_dict(torch.load(CLASSIFIER_WEIGHTS, map_location=DEVICE))
    resnet_model = resnet_model.to(DEVICE).eval()

    segmenter = HieroglyphSegmenter(sam_checkpoint_path=SAM_CHECKPOINT)
    classifier = HieroglyphClassifier(model=resnet_model, idx_to_class=idx_to_class, transform=inference_transforms)
    nlp_pipeline = ExactCorpusMatchPipeline(lexicon_path=LEXICON_PATH)

    # 2. Run Computer Vision Segmentation and Sorter
    if os.path.exists(TARGET_IMAGE):
        orig_img, unified_mask, crops, prompts, valid_cnts = segmenter.extract_symbols(TARGET_IMAGE)
        print(f"SAM Segmentation Yield: {valid_cnts} localized symbols out of {prompts} anchor points.")

        # 3. Classify and Remap Codes
        raw_detected_sequence = []
        for crop, bbox in crops:
            prediction = classifier.classify(crop)
            if prediction['confidence'] >= 0.25:
                raw_detected_sequence.append(prediction['label'])

        # Enforce strict 'I' -> 'J' string conversions natively across generated codes
        final_sequence_list = [label.replace('I', 'J') for label in raw_detected_sequence]
        
        print("\n" + "="*70)
        print("🚀 STEP 1 COMPLETE: COMPUTER VISION EXTRACTION VALUE:")
        print(f"final_sequence_list = {final_sequence_list}")
        print("="*70)

        # 4. Pass Array Variable into the Translation NLP Engine
        print("\n" + "="*70)
        print("🚀 STEP 2 RUNNING: NLP LEXICON SLIDING-WINDOW LOOKUP:")
        print("="*70)
        
        translation_results = nlp_pipeline.process_gardiner_list(final_sequence_list)
        
        print(f"Lookup Execution Strategy :  {translation_results['strategy'].upper()}")
        print(f"OpenNMT Corpus Text (.char):  {translation_results['open_nmt_ready_string']}")
        print(f"Linguistic Meaning Hint   :  {translation_results['english_hint']}")
        print("="*70 + "\n")
    else:
        print(f"❌ Aborted: Could not locate your project target image space at: {TARGET_IMAGE}")